## Begin

In [ ]:
# ============================================================
# Notebook Notes:
# ============================================================
# This notebook reproduces the semi-synthetic IHDP experiment for
# IntervalGP-VAE and compares it with a conformalized TEDVAE baseline.
#
# Main goal:
#   Evaluate IntervalGP-VAE on the IHDP benchmark across all 100
#   replications and compare its ITE estimation accuracy, ATE error,
#   empirical interval coverage, interval length, and runtime against
#   TEDVAE.
#
# Dataset:
#   The experiment uses the IHDP semi-synthetic benchmark.
#
#   Data are loaded from:
#      ./IHDP_b/
#
#   The notebook runs:
#      SEARCH_REPS = 1, ..., 100
#
# Proxy variables:
#   Only continuous IHDP covariates are used as proxy variables Z.
#
#   Binary covariates are excluded from the proxy input in this
#   implementation.
#
# Compared methods:
#   1) IntervalGP-VAE
#      - Uses proxy-based latent confounder recovery.
#      - Uses local IntervalGP regularization during training.
#      - Uses latent GP posterior smoothing and ITE-space FullGP
#        posterior inference.
#      - Produces model-based 90% ITE intervals.
#
#   2) Conformalized TEDVAE
#      - Uses TEDVAE as the point ITE estimator.
#      - Splits the training data into fitting and calibration sets.
#      - Uses calibration residuals to construct conformal 90% ITE
#        intervals.
#
# IntervalGP-VAE configuration:
#   config_id = 486
#   chosen_version = "u_aux"
#   latent_dim = 4
#   hidden_dim = 64
#   head_hidden_dim = 192
#
# IntervalGP-VAE GP parameters:
#   gp_lengthscale = 1.85
#   gp_variance = 3
#   gp_noise = 0.012
#
# IntervalGP-VAE loss weights:
#   kl_u_scale = 0.005
#   kl_eps_scale = 0.5
#   kl_aux_scale = 0.5
#   std_penalty_weight = 0.25
#
# IntervalGP-VAE training configuration:
#   batch_size = 128
#   joint_epochs = 650
#   head_epochs = 250
#   vae_refine_epochs = 75
#
# Optimizer:
#   AdamW is used with:
#      lr_joint = 1e-3
#      lr_head = 1e-3
#      lr_vae_refine = 1e-4
#      weight_decay = 1e-5
#
# Three-stage IntervalGP-VAE training:
#   Stage 1:
#      Jointly train the VAE and causal head.
#
#   Stage 2:
#      Freeze the VAE and train only the causal head using the
#      encoder posterior mean.
#
#   Stage 3:
#      Freeze the causal head and refine the VAE using the causal loss.
#
# ITE uncertainty:
#   ITE_CI = 0.90
#   Z_VALUE_90 = 1.6448536269514722
#   n_samples_ite = 100
#
# GP reference setting:
#   MAX_GP_POINTS = None
#
# Data preprocessing:
#   standardize_z = True
#      Z is standardized using training-set statistics.
#
#   standardize_y = False
#      Y is not standardized in this configuration.
#
# TEDVAE configuration:
#   latent_dim = 20
#   latent_dim_t = 10
#   latent_dim_y = 10
#   hidden_dim = 500
#   num_layers = 4
#   num_samples = 10
#   num_epochs = 200
#   batch_size = 1000
#   learning_rate = 1e-3
#   learning_rate_decay = 0.01
#   weight_decay = 1e-4
#
# True ITE:
#   The true ITE is provided by the IHDP benchmark and is used for
#   PEHE, ATE error, and interval-coverage evaluation.
#
# Evaluation metrics:
#   PEHE
#   ATE
#   ATE error
#   90% empirical ITE coverage
#   Average ITE interval length
#   Runtime
#
# Per-replication outputs:
#   For each IHDP replication, the notebook saves:
#      true ITE
#      IntervalGP-VAE ITE mean/lower/upper
#      TEDVAE ITE mean/lower/upper
#
#   File pattern:
#      ihdp_rep_<rep_id>_config_results.csv
#
# Summary outputs:
#   Partial summary:
#      config_intervalgpvae_vs_tedvae_summary_partial.csv
#
#   Final summary:
#      config_intervalgpvae_vs_tedvae_summary_final.csv
#
# Plot outputs:
#   The notebook saves comparison plots for:
#      PEHE
#      ATE error
#      ITE interval coverage
#      ITE interval length
#
#   Plot files:
#      config_ihdp_pehe_comparison.png
#      config_ihdp_ate_error_comparison.png
#      config_ihdp_coverage_comparison.png
#      config_ihdp_interval_length_comparison.png
#
# Output directory:
#   ihdp_reproduce_intervalgpvae_vs_tedvae
#
# Purpose:
#   This experiment provides the semi-synthetic IHDP evaluation for
#   the TMLR revision. It tests whether IntervalGP-VAE can achieve
#   competitive ITE and ATE accuracy while producing substantially
#   shorter model-based uncertainty intervals than conformalized TEDVAE.
# ============================================================

In [ ]:
# =========================================================
# Cell 1: Imports and basic settings
# =========================================================
import os
import time
import json
import pyro
import torch
import random
import logging
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from copy import deepcopy
from torch.utils.data import TensorDataset, DataLoader

from causal_datasets import IHDP
from tedvae_gpu import TEDVAE


logging.getLogger("pyro").setLevel(logging.ERROR)
pyro.enable_validation(True)

torch.set_default_tensor_type("torch.FloatTensor")


# =========================================================
# Cell 2: Global settings
# =========================================================

DEVICE = "cpu"

GLOBAL_SEED = 420

OUTPUT_DIR = "ihdp_reproduce_intervalgpvae_vs_tedvae"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Run all 100 IHDP replications
SEARCH_REPS = list(range(1, 101))

# For quick debugging, use:
# SEARCH_REPS = [1, 2]

PLOT_EACH_REP = False

ITE_CI = 0.90
Z_VALUE_90 = 1.6448536269514722

# Full GP over all training samples
MAX_GP_POINTS = None


CFG = {
    "config_id": 486,

    # Model type
    "chosen_version": "u_aux",

    # Model size
    "latent_dim": 4,
    "hidden_dim": 64,
    "head_hidden_dim": 192,

    # GP parameters
    "gp_lengthscale": 1.85,
    "gp_variance": 3,
    "gp_noise": 0.012,

    # VAE loss weights
    "kl_u_scale": 0.005,
    "kl_eps_scale": 0.5,
    "kl_aux_scale": 0.5,
    "std_penalty_weight": 0.25,

    # Training
    "batch_size": 128,
    "joint_epochs": 650,
    "head_epochs": 250,
    "vae_refine_epochs": 75,

    "lr_joint": 1e-3,
    "lr_head": 1e-3,
    "lr_vae_refine": 1e-4,
    "weight_decay": 1e-5,

    # Inference
    "n_samples_ite": 100,
    "ite_ci": 0.90,
    "z_value": 1.6448536269514722,

    # Stabilization
    "grad_clip": None,

    # Data preprocessing
    "standardize_z": True,
    "standardize_y": False,
}

print(json.dumps(CFG, indent=4))


# =========================================================
# Cell 4: TEDVAE settings
# =========================================================

TEDVAE_ARGS = {
    "latent_dim": 20,
    "latent_dim_t": 10,
    "latent_dim_y": 10,
    "hidden_dim": 500,
    "num_layers": 4,
    "num_samples": 10,
    "num_epochs": 200,
    "batch_size": 1000,
    "learning_rate": 1e-3,
    "learning_rate_decay": 0.01,
    "weight_decay": 1e-4,
}


# =========================================================
# Cell 5: Utility functions
# =========================================================

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    pyro.set_rng_seed(seed)


def to_numpy_1d(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    return np.asarray(x).reshape(-1)


def pehe_np(est_ite, true_ite):
    est_ite = to_numpy_1d(est_ite)
    true_ite = to_numpy_1d(true_ite)

    valid = np.isfinite(est_ite) & np.isfinite(true_ite)

    if valid.sum() == 0:
        return np.nan

    return float(
        np.sqrt(
            np.mean(
                (est_ite[valid] - true_ite[valid]) ** 2
            )
        )
    )


def ate_error_np(est_ite, true_ite):
    est_ite = to_numpy_1d(est_ite)
    true_ite = to_numpy_1d(true_ite)

    return float(
        abs(
            np.nanmean(est_ite) - np.nanmean(true_ite)
        )
    )


def standardize_train_test(z_train, z_test):
    z_mean = z_train.mean(dim=0, keepdim=True)
    z_std = z_train.std(dim=0, keepdim=True).clamp_min(1e-6)

    z_train_std = (z_train - z_mean) / z_std
    z_test_std = (z_test - z_mean) / z_std

    return z_train_std, z_test_std, z_mean, z_std


def standardize_y_train(y_train):
    y_mean = y_train.mean()
    y_std = y_train.std().clamp_min(1e-6)
    y_train_std = (y_train - y_mean) / y_std

    return y_train_std, y_mean, y_std


def split_train_calibration_indices(
    n,
    calibration_ratio=0.2,
    seed=123,
):
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)

    n_cal = max(1, int(n * calibration_ratio))

    cal_idx = idx[:n_cal]
    fit_idx = idx[n_cal:]

    return fit_idx, cal_idx


def conformal_abs_residual_quantile(
    y_true_cal,
    y_pred_cal,
    alpha=0.10,
):
    y_true_cal = to_numpy_1d(y_true_cal)
    y_pred_cal = to_numpy_1d(y_pred_cal)

    residuals = np.abs(y_true_cal - y_pred_cal)
    residuals = residuals[np.isfinite(residuals)]

    n = len(residuals)

    if n == 0:
        return np.nan

    q_level = np.ceil((n + 1) * (1 - alpha)) / n
    q_level = min(q_level, 1.0)

    return float(
        np.quantile(
            residuals,
            q_level,
            method="higher",
        )
    )


def make_conformal_interval(point_pred, q_hat):
    point_pred = to_numpy_1d(point_pred)

    lower = point_pred - q_hat
    upper = point_pred + q_hat

    return lower, upper


def compute_interval_metrics(lower, upper, true_value):
    lower = to_numpy_1d(lower)
    upper = to_numpy_1d(upper)
    true_value = to_numpy_1d(true_value)

    lo = np.minimum(lower, upper)
    hi = np.maximum(lower, upper)

    valid = (
        np.isfinite(lo)
        & np.isfinite(hi)
        & np.isfinite(true_value)
    )

    if valid.sum() == 0:
        return {
            "coverage": np.nan,
            "avg_length": np.nan,
            "n": 0,
        }

    covered = (
        (lo[valid] <= true_value[valid])
        & (true_value[valid] <= hi[valid])
    )

    return {
        "coverage": float(np.mean(covered)),
        "avg_length": float(np.mean(hi[valid] - lo[valid])),
        "n": int(valid.sum()),
    }


def maybe_subsample_gp_reference(
    x,
    *ys,
    max_points=None,
    seed=0,
):
    n = x.shape[0]

    if max_points is None or n <= max_points:
        return (x, *ys)

    rng = np.random.default_rng(seed)
    idx = rng.choice(
        np.arange(n),
        size=max_points,
        replace=False,
    )

    idx = np.sort(idx)

    idx_t = torch.tensor(
        idx,
        dtype=torch.long,
        device=x.device,
    )

    out = [x[idx_t]]

    for y in ys:
        if torch.is_tensor(y):
            out.append(y[idx_t])
        else:
            out.append(np.asarray(y)[idx])

    return tuple(out)


# =========================================================
# Cell 6: GP posterior functions
# =========================================================

def rbf_kernel(
    x1,
    x2=None,
    lengthscale=1.0,
    variance=1.0,
):
    if x2 is None:
        x2 = x1

    if x1.ndim == 1:
        x1 = x1.view(-1, 1)

    if x2.ndim == 1:
        x2 = x2.view(-1, 1)

    dists = torch.cdist(x1, x2).pow(2)

    return variance * torch.exp(
        -0.5 * dists / (lengthscale ** 2)
    )


def full_gp_posterior_point(
    x_train,
    y_train,
    x_test,
    lengthscale=1.0,
    variance=1.0,
    noise=1e-2,
    jitter=1e-5,
):
    x_train = x_train.float()
    x_test = x_test.float()
    y_train = y_train.float().view(-1, 1)

    n_train = x_train.shape[0]
    n_test = x_test.shape[0]

    K = rbf_kernel(
        x_train,
        x_train,
        lengthscale=lengthscale,
        variance=variance,
    )

    K = K + noise * torch.eye(
        n_train,
        device=x_train.device,
    )

    K_s = rbf_kernel(
        x_train,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = rbf_kernel(
        x_test,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = K_ss + jitter * torch.eye(
        n_test,
        device=x_test.device,
    )

    current_jitter = jitter

    for _ in range(8):
        try:
            L = torch.linalg.cholesky(
                K
                + current_jitter
                * torch.eye(n_train, device=x_train.device)
            )
            break

        except RuntimeError:
            current_jitter *= 10

    else:
        raise RuntimeError(
            "Cholesky failed in full_gp_posterior_point."
        )

    alpha = torch.cholesky_solve(y_train, L)

    posterior_mean = K_s.t() @ alpha

    v = torch.cholesky_solve(K_s, L)
    posterior_cov = K_ss - K_s.t() @ v
    posterior_cov = 0.5 * (posterior_cov + posterior_cov.t())

    return posterior_mean.squeeze(-1), posterior_cov


def full_gp_posterior_heteroscedastic(
    x_train,
    y_train,
    x_test,
    noise_var_train,
    lengthscale=1.0,
    variance=1.0,
    min_noise=1e-2,
    jitter=1e-5,
):
    x_train = x_train.float()
    x_test = x_test.float()
    y_train = y_train.float().view(-1, 1)

    noise_var_train = noise_var_train.float().view(-1)
    noise_var_train = torch.clamp(
        noise_var_train,
        min=min_noise,
    )

    n_train = x_train.shape[0]
    n_test = x_test.shape[0]

    K = rbf_kernel(
        x_train,
        x_train,
        lengthscale=lengthscale,
        variance=variance,
    )

    K = K + torch.diag(noise_var_train)

    K_s = rbf_kernel(
        x_train,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = rbf_kernel(
        x_test,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = K_ss + jitter * torch.eye(
        n_test,
        device=x_test.device,
    )

    current_jitter = jitter

    for _ in range(8):
        try:
            L = torch.linalg.cholesky(
                K
                + current_jitter
                * torch.eye(n_train, device=x_train.device)
            )
            break

        except RuntimeError:
            current_jitter *= 10

    else:
        raise RuntimeError(
            "Cholesky failed in full_gp_posterior_heteroscedastic."
        )

    alpha = torch.cholesky_solve(y_train, L)

    posterior_mean = K_s.t() @ alpha

    v = torch.cholesky_solve(K_s, L)
    posterior_cov = K_ss - K_s.t() @ v
    posterior_cov = 0.5 * (posterior_cov + posterior_cov.t())

    return posterior_mean.squeeze(-1), posterior_cov


def latent_gp_posterior_smoothing(
    z_train,
    mu_u_train,
    z_test,
    cfg,
):
    if mu_u_train.ndim == 1:
        mu_u_train = mu_u_train.view(-1, 1)

    latent_dim = mu_u_train.shape[1]

    means = []
    vars_diag = []

    for r in range(latent_dim):
        mean_r, cov_r = full_gp_posterior_point(
            x_train=z_train,
            y_train=mu_u_train[:, r],
            x_test=z_test,
            lengthscale=cfg["gp_lengthscale"],
            variance=cfg["gp_variance"],
            noise=cfg["gp_noise"],
        )

        var_r = torch.diag(cov_r).clamp_min(1e-8)

        means.append(mean_r)
        vars_diag.append(var_r)

    mean = torch.stack(means, dim=1)
    std = torch.sqrt(
        torch.stack(vars_diag, dim=1)
    )

    return mean, std


# =========================================================
# Cell 7: IntervalGP-VAE model
# =========================================================

LOGVAR_MIN = -10.0
LOGVAR_MAX = 6.0


class DoubleEncoder(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        self.fc1 = nn.Linear(input_dim, hidden_dim)

        self.mean_u = nn.Linear(hidden_dim, latent_dim)
        self.logvar_u = nn.Linear(hidden_dim, latent_dim)

        self.mean_eps = nn.Linear(hidden_dim, input_dim)
        self.logvar_eps = nn.Linear(hidden_dim, input_dim)

        if self.use_aux:
            self.mean_ua0 = nn.Linear(hidden_dim, latent_dim)
            self.logvar_ua0 = nn.Linear(hidden_dim, latent_dim)

            self.mean_ua1 = nn.Linear(hidden_dim, latent_dim)
            self.logvar_ua1 = nn.Linear(hidden_dim, latent_dim)

    def forward(self, z):
        h = F.relu(self.fc1(z))

        mu_u = self.mean_u(h)

        logvar_u = torch.clamp(
            self.logvar_u(h),
            min=LOGVAR_MIN,
            max=LOGVAR_MAX,
        )

        std_u = torch.exp(0.5 * logvar_u)

        mu_eps = self.mean_eps(h)

        logvar_eps = torch.clamp(
            self.logvar_eps(h),
            min=LOGVAR_MIN,
            max=LOGVAR_MAX,
        )

        std_eps = torch.exp(0.5 * logvar_eps)

        if self.use_aux:
            mu_ua0 = self.mean_ua0(h)

            logvar_ua0 = torch.clamp(
                self.logvar_ua0(h),
                min=LOGVAR_MIN,
                max=LOGVAR_MAX,
            )

            std_ua0 = torch.exp(0.5 * logvar_ua0)

            mu_ua1 = self.mean_ua1(h)

            logvar_ua1 = torch.clamp(
                self.logvar_ua1(h),
                min=LOGVAR_MIN,
                max=LOGVAR_MAX,
            )

            std_ua1 = torch.exp(0.5 * logvar_ua1)

            return (
                mu_u, std_u, logvar_u,
                mu_eps, std_eps, logvar_eps,
                mu_ua0, std_ua0, logvar_ua0,
                mu_ua1, std_ua1, logvar_ua1,
            )

        return (
            mu_u, std_u, logvar_u,
            mu_eps, std_eps, logvar_eps,
        )


class AdditiveDecoder(nn.Module):
    def __init__(
        self,
        latent_dim,
        hidden_dim,
        output_dim,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        factor = 1 + (2 if use_auxiliary_latents else 0)

        self.fc1 = nn.Linear(
            latent_dim * factor,
            hidden_dim,
        )

        self.out = nn.Linear(hidden_dim, output_dim)

    def forward(self, u, eps, ua0=None, ua1=None):
        if self.use_aux:
            x = torch.cat([u, ua0, ua1], dim=1)
        else:
            x = u

        h = F.relu(self.fc1(x))

        return self.out(h) + eps


class GPVAEwithNoise(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        gp_lengthscale=2.0,
        gp_variance=2.0,
        gp_noise=1e-2,
        use_auxiliary_latents=False,
        kl_u_scale=0.001,
        kl_eps_scale=0.5,
        kl_aux_scale=0.5,
        std_penalty_weight=0.5,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        self.encoder = DoubleEncoder(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

        self.decoder = AdditiveDecoder(
            latent_dim=latent_dim,
            hidden_dim=hidden_dim,
            output_dim=input_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

        self.gp_lengthscale = gp_lengthscale
        self.gp_variance = gp_variance
        self.gp_noise = gp_noise

        self.kl_u_scale = kl_u_scale
        self.kl_eps_scale = kl_eps_scale
        self.kl_aux_scale = kl_aux_scale
        self.std_penalty_weight = std_penalty_weight

    def reparameterize(self, mu, std):
        return mu + torch.randn_like(std) * std

    def local_interval_gp_regularizer(
        self,
        z,
        mu_u,
        std_u,
    ):
        k_ii = self.gp_variance * torch.ones_like(mu_u)

        gp_std = torch.sqrt(
            k_ii + self.gp_noise
        )

        lower = mu_u - std_u
        upper = mu_u + std_u

        normal = torch.distributions.Normal(
            loc=torch.zeros_like(mu_u),
            scale=gp_std,
        )

        prob = normal.cdf(upper) - normal.cdf(lower)
        prob = torch.clamp(prob, min=1e-8)

        return -torch.sum(torch.log(prob))

    def get_latent_stats(self, z, var_name="u"):
        outputs = self.encoder(z)

        if var_name == "u":
            return outputs[0], outputs[1], outputs[2]

        if var_name == "eps":
            return outputs[3], outputs[4], outputs[5]

        if var_name == "ua0":
            if not self.use_aux:
                return None, None, None

            return outputs[6], outputs[7], outputs[8]

        if var_name == "ua1":
            if not self.use_aux:
                return None, None, None

            return outputs[9], outputs[10], outputs[11]

        raise ValueError(f"Invalid var_name: {var_name}")

    def sample_latent(self, z, var_name="u"):
        mu, std, _ = self.get_latent_stats(
            z,
            var_name=var_name,
        )

        if mu is None:
            return None

        return self.reparameterize(mu, std)

    def forward(self, z):
        if self.use_aux:
            (
                mu_u, std_u, logvar_u,
                mu_eps, std_eps, logvar_eps,
                mu_ua0, std_ua0, logvar_ua0,
                mu_ua1, std_ua1, logvar_ua1,
            ) = self.encoder(z)

            u = self.reparameterize(mu_u, std_u)
            eps = self.reparameterize(mu_eps, std_eps)
            ua0 = self.reparameterize(mu_ua0, std_ua0)
            ua1 = self.reparameterize(mu_ua1, std_ua1)

            z_recon = self.decoder(
                u,
                eps,
                ua0=ua0,
                ua1=ua1,
            )

        else:
            (
                mu_u, std_u, logvar_u,
                mu_eps, std_eps, logvar_eps,
            ) = self.encoder(z)

            u = self.reparameterize(mu_u, std_u)
            eps = self.reparameterize(mu_eps, std_eps)

            ua0 = None
            ua1 = None

            z_recon = self.decoder(u, eps)

        recon_loss = F.mse_loss(
            z_recon,
            z,
            reduction="sum",
        )

        kl_u = -self.kl_u_scale * torch.sum(
            1 + logvar_u - mu_u.pow(2) - logvar_u.exp()
        )

        kl_eps = -self.kl_eps_scale * torch.sum(
            1 + logvar_eps - mu_eps.pow(2) - logvar_eps.exp()
        )

        gp_interval_reg = self.local_interval_gp_regularizer(
            z=z,
            mu_u=mu_u,
            std_u=std_u,
        )

        std_penalty = torch.sum(std_u ** 2)

        total_loss = (
            recon_loss
            + kl_u
            + kl_eps
            + gp_interval_reg
            + self.std_penalty_weight * std_penalty
        )

        if self.use_aux:
            kl_ua0 = -self.kl_aux_scale * torch.sum(
                1 + logvar_ua0 - mu_ua0.pow(2) - logvar_ua0.exp()
            )

            kl_ua1 = -self.kl_aux_scale * torch.sum(
                1 + logvar_ua1 - mu_ua1.pow(2) - logvar_ua1.exp()
            )

            total_loss = total_loss + kl_ua0 + kl_ua1

        return total_loss, {
            "recon_loss": recon_loss.item(),
            "kl_u": kl_u.item(),
            "kl_eps": kl_eps.item(),
            "gp_interval_reg": gp_interval_reg.item(),
            "u": u,
            "eps": eps,
            "ua0": ua0,
            "ua1": ua1,
            "mu_u": mu_u,
            "std_u": std_u,
            "z_recon": z_recon,
        }


class FlexibleCausalHead(nn.Module):
    def __init__(
        self,
        latent_dim_u,
        hidden_dim,
        z_y_dim=None,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents
        self.z_y_dim = z_y_dim

        input_dim = latent_dim_u + 1

        if z_y_dim is not None:
            input_dim += z_y_dim

        if use_auxiliary_latents:
            input_dim += latent_dim_u

        self.fc = nn.Linear(input_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, u, t, z_y=None, ua1=None):
        t = t.view(-1, 1)

        parts = [u, t]

        if self.z_y_dim is not None:
            if z_y is None:
                raise ValueError("Expected z_y input but got None.")

            parts.append(z_y)

        if self.use_aux:
            if ua1 is None:
                raise ValueError("Expected ua1 input but got None.")

            parts.append(ua1)

        x = torch.cat(parts, dim=1)

        h = F.relu(self.fc(x))

        return self.out(h)


class CausalGPVAEwithNoise(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        head_hidden_dim,
        z_y_dim=None,
        use_auxiliary_latents=False,
        gp_lengthscale=2.0,
        gp_variance=2.0,
        gp_noise=1e-2,
        kl_u_scale=0.001,
        kl_eps_scale=0.5,
        kl_aux_scale=0.5,
        std_penalty_weight=0.5,
    ):
        super().__init__()

        self.z_y_dim = z_y_dim
        self.use_aux = use_auxiliary_latents

        self.vae = GPVAEwithNoise(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            gp_lengthscale=gp_lengthscale,
            gp_variance=gp_variance,
            gp_noise=gp_noise,
            use_auxiliary_latents=use_auxiliary_latents,
            kl_u_scale=kl_u_scale,
            kl_eps_scale=kl_eps_scale,
            kl_aux_scale=kl_aux_scale,
            std_penalty_weight=std_penalty_weight,
        )

        self.causal_head = FlexibleCausalHead(
            latent_dim_u=latent_dim,
            hidden_dim=head_hidden_dim,
            z_y_dim=z_y_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

    def forward(self, z, t, y):
        loss_vae, vae_info = self.vae(z)

        u = vae_info["u"]
        ua1 = vae_info["ua1"] if self.use_aux else None

        z_y = make_z_y(self, z)

        y_pred = self.causal_head(
            u,
            t,
            z_y=z_y,
            ua1=ua1,
        )

        causal_loss = F.mse_loss(
            y_pred,
            y.unsqueeze(-1),
            reduction="sum",
        )

        total_loss = loss_vae + causal_loss

        return total_loss, {
            **vae_info,
            "y_pred": y_pred,
            "causal_loss": causal_loss.item(),
        }


def make_z_y(model, z):
    if model.causal_head.z_y_dim == z.shape[1]:
        return z

    if model.causal_head.z_y_dim:
        return z[:, z.shape[1] // 2:]

    return None


# =========================================================
# Cell 8: Training procedure for IntervalGP-VAE config
# =========================================================

def train_intervalgpvae_config(
    model,
    loader,
    cfg,
    device=DEVICE,
):
    # -----------------------------------------------------
    # Stage 1: joint training
    # -----------------------------------------------------
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg["lr_joint"],
        weight_decay=cfg["weight_decay"],
    )

    for epoch in range(cfg["joint_epochs"]):
        model.train()

        for batch in loader:
            z_batch, t_batch, y_batch = [
                b.to(device) for b in batch
            ]

            loss, info = model(
                z_batch,
                t_batch,
                y_batch,
            )

            optimizer.zero_grad()
            loss.backward()

            if cfg["grad_clip"] is not None:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=cfg["grad_clip"],
                )

            optimizer.step()

    # -----------------------------------------------------
    # Stage 2: freeze VAE and train causal head
    # -----------------------------------------------------
    for param in model.vae.parameters():
        param.requires_grad = False

    for param in model.causal_head.parameters():
        param.requires_grad = True

    causal_optimizer = torch.optim.AdamW(
        model.causal_head.parameters(),
        lr=cfg["lr_head"],
        weight_decay=cfg["weight_decay"],
    )

    for epoch in range(cfg["head_epochs"]):
        model.train()

        for batch in loader:
            z_batch, t_batch, y_batch = [
                b.to(device) for b in batch
            ]

            with torch.no_grad():
                mu_u = model.vae.get_latent_stats(
                    z_batch,
                    var_name="u",
                )[0]

                if model.use_aux:
                    mu_ua1, std_ua1, _ = model.vae.get_latent_stats(
                        z_batch,
                        var_name="ua1",
                    )

                    ua1_batch = model.vae.reparameterize(
                        mu_ua1,
                        std_ua1,
                    )

                else:
                    ua1_batch = None

            z_y_batch = make_z_y(model, z_batch)

            y_pred = model.causal_head(
                mu_u.detach(),
                t_batch,
                z_y=z_y_batch,
                ua1=ua1_batch.detach() if ua1_batch is not None else None,
            )

            causal_loss = F.mse_loss(
                y_pred,
                y_batch.unsqueeze(-1),
            )

            causal_optimizer.zero_grad()
            causal_loss.backward()
            causal_optimizer.step()

    # -----------------------------------------------------
    # Stage 3: freeze causal head and refine VAE
    # -----------------------------------------------------
    for param in model.causal_head.parameters():
        param.requires_grad = False

    for param in model.vae.parameters():
        param.requires_grad = True

    vae_optimizer = torch.optim.AdamW(
        model.vae.parameters(),
        lr=cfg["lr_vae_refine"],
        weight_decay=cfg["weight_decay"],
    )

    for epoch in range(cfg["vae_refine_epochs"]):
        model.train()

        for batch in loader:
            z_batch, t_batch, y_batch = [
                b.to(device) for b in batch
            ]

            z_y_batch = make_z_y(model, z_batch)

            loss_vae, vae_info = model.vae(z_batch)

            ua1_batch = (
                model.vae.sample_latent(
                    z_batch,
                    var_name="ua1",
                )
                if model.use_aux
                else None
            )

            y_pred = model.causal_head(
                vae_info["u"],
                t_batch,
                z_y=z_y_batch,
                ua1=ua1_batch,
            )

            causal_loss = F.mse_loss(
                y_pred,
                y_batch.unsqueeze(-1),
                reduction="sum",
            )

            total_loss = loss_vae + causal_loss

            vae_optimizer.zero_grad()
            total_loss.backward()

            if cfg["grad_clip"] is not None:
                torch.nn.utils.clip_grad_norm_(
                    model.vae.parameters(),
                    max_norm=cfg["grad_clip"],
                )

            vae_optimizer.step()

    for param in model.parameters():
        param.requires_grad = True

    return model


# =========================================================
# Cell 9: ITE sampling and ITE-space IntervalGP head
# =========================================================

def sample_ite_from_encoder(
    model,
    z,
    cfg,
):
    model.eval()

    n_samples = cfg["n_samples_ite"]
    ite_ci = cfg["ite_ci"]

    with torch.no_grad():
        mu_u, std_u, _ = model.vae.get_latent_stats(
            z,
            var_name="u",
        )

        N, d_u = mu_u.shape
        device = z.device

        z_y = make_z_y(model, z)

        mu_ua1, std_ua1, _ = model.vae.get_latent_stats(
            z,
            var_name="ua1",
        )

        has_ua1 = (
            mu_ua1 is not None
            and std_ua1 is not None
        )

        eps_u = torch.randn(
            n_samples,
            N,
            d_u,
            device=device,
        )

        u_samples = mu_u.unsqueeze(0) + eps_u * std_u.unsqueeze(0)

        U = u_samples.reshape(-1, d_u)

        if has_ua1:
            d_ua = mu_ua1.shape[1]

            eps_ua1 = torch.randn(
                n_samples,
                N,
                d_ua,
                device=device,
            )

            ua1_samples = (
                mu_ua1.unsqueeze(0)
                + eps_ua1 * std_ua1.unsqueeze(0)
            )

            UA1 = ua1_samples.reshape(-1, d_ua)

        else:
            UA1 = None

        if z_y is not None:
            ZY = (
                z_y.unsqueeze(0)
                .expand(n_samples, -1, -1)
                .reshape(-1, z_y.shape[1])
            )

        else:
            ZY = None

        t0 = torch.zeros(U.shape[0], device=device)
        t1 = torch.ones(U.shape[0], device=device)

        y0 = model.causal_head(
            U,
            t0,
            z_y=ZY,
            ua1=UA1,
        ).squeeze(-1)

        y1 = model.causal_head(
            U,
            t1,
            z_y=ZY,
            ua1=UA1,
        ).squeeze(-1)

        ite_samples = (y1 - y0).view(n_samples, N)

        lower_q = (1.0 - ite_ci) / 2.0
        upper_q = 1.0 - lower_q

        ite_mean = ite_samples.mean(dim=0)
        ite_std = ite_samples.std(dim=0)

        ite_lower = ite_samples.quantile(
            lower_q,
            dim=0,
        )

        ite_upper = ite_samples.quantile(
            upper_q,
            dim=0,
        )

        return (
            ite_samples,
            ite_mean,
            ite_std,
            ite_lower,
            ite_upper,
        )


def ite_space_intervalgp_head(
    model,
    z_train,
    z_test,
    cfg,
):
    model.eval()

    with torch.no_grad():
        (
            _,
            ite_mean_train,
            ite_std_train,
            ite_lower_train,
            ite_upper_train,
        ) = sample_ite_from_encoder(
            model=model,
            z=z_train,
            cfg=cfg,
        )

        mu_u_train, _, _ = model.vae.get_latent_stats(
            z_train,
            var_name="u",
        )

        mu_u_test, _, _ = model.vae.get_latent_stats(
            z_test,
            var_name="u",
        )

        u_test_smooth, u_test_smooth_std = latent_gp_posterior_smoothing(
            z_train=z_train,
            mu_u_train=mu_u_train,
            z_test=z_test,
            cfg=cfg,
        )

        u_train_smooth = mu_u_train.detach()

        (
            u_train_gp,
            ite_lower_train_gp,
            ite_upper_train_gp,
            ite_mean_train_gp,
            ite_std_train_gp,
        ) = maybe_subsample_gp_reference(
            u_train_smooth,
            ite_lower_train.detach(),
            ite_upper_train.detach(),
            ite_mean_train.detach(),
            ite_std_train.detach(),
            max_points=MAX_GP_POINTS,
            seed=123,
        )

        noise_var_lower = ite_std_train_gp.pow(2) + cfg["gp_noise"]
        noise_var_upper = ite_std_train_gp.pow(2) + cfg["gp_noise"]
        noise_var_mean = ite_std_train_gp.pow(2) + cfg["gp_noise"]

        gp_lower_mean, gp_lower_cov = full_gp_posterior_heteroscedastic(
            x_train=u_train_gp,
            y_train=ite_lower_train_gp,
            x_test=u_test_smooth,
            noise_var_train=noise_var_lower,
            lengthscale=cfg["gp_lengthscale"],
            variance=cfg["gp_variance"],
            min_noise=cfg["gp_noise"],
        )

        gp_upper_mean, gp_upper_cov = full_gp_posterior_heteroscedastic(
            x_train=u_train_gp,
            y_train=ite_upper_train_gp,
            x_test=u_test_smooth,
            noise_var_train=noise_var_upper,
            lengthscale=cfg["gp_lengthscale"],
            variance=cfg["gp_variance"],
            min_noise=cfg["gp_noise"],
        )

        gp_ite_mean, gp_ite_cov = full_gp_posterior_heteroscedastic(
            x_train=u_train_gp,
            y_train=ite_mean_train_gp,
            x_test=u_test_smooth,
            noise_var_train=noise_var_mean,
            lengthscale=cfg["gp_lengthscale"],
            variance=cfg["gp_variance"],
            min_noise=cfg["gp_noise"],
        )

        gp_lower_std = torch.sqrt(
            torch.diag(gp_lower_cov).clamp_min(1e-8)
        )

        gp_upper_std = torch.sqrt(
            torch.diag(gp_upper_cov).clamp_min(1e-8)
        )

        gp_ite_std = torch.sqrt(
            torch.diag(gp_ite_cov).clamp_min(1e-8)
        )

        raw_lower = torch.minimum(
            gp_lower_mean,
            gp_upper_mean,
        )

        raw_upper = torch.maximum(
            gp_lower_mean,
            gp_upper_mean,
        )

        ite_lower_test = raw_lower - cfg["z_value"] * gp_lower_std
        ite_upper_test = raw_upper + cfg["z_value"] * gp_upper_std

        return {
            "ite_mean_test": gp_ite_mean,
            "ite_std_test": gp_ite_std,
            "ite_lower_test": ite_lower_test,
            "ite_upper_test": ite_upper_test,
            "u_test_smooth": u_test_smooth,
            "u_test_smooth_std": u_test_smooth_std,
            "u_test_raw": mu_u_test,
            "ite_lower_train": ite_lower_train,
            "ite_upper_train": ite_upper_train,
            "ite_mean_train": ite_mean_train,
            "ite_std_train": ite_std_train,
        }


# =========================================================
# Cell 10: Plotting function
# =========================================================

def plot_ite_interval(
    true_ite,
    ite_point,
    ite_lower,
    ite_upper,
    iteration_id,
    method_name,
    output_dir=OUTPUT_DIR,
):
    true_ite_np = to_numpy_1d(true_ite)
    ite_point_np = to_numpy_1d(ite_point)
    ite_lower_np = to_numpy_1d(ite_lower)
    ite_upper_np = to_numpy_1d(ite_upper)

    x = np.arange(len(true_ite_np))

    metrics = compute_interval_metrics(
        lower=ite_lower_np,
        upper=ite_upper_np,
        true_value=true_ite_np,
    )

    plt.figure(figsize=(11, 5))

    plt.fill_between(
        x,
        ite_lower_np,
        ite_upper_np,
        alpha=0.25,
        label=f"{method_name} ITE interval",
    )

    plt.plot(
        x,
        ite_point_np,
        linestyle="--",
        linewidth=1.5,
        alpha=0.75,
        label=f"{method_name} point ITE",
    )

    plt.plot(
        x,
        true_ite_np,
        linestyle="-",
        marker=".",
        linewidth=1.3,
        alpha=0.80,
        label="True ITE",
    )

    plt.text(
        0.02,
        0.96,
        f"coverage = {metrics['coverage']:.3f}\n"
        f"avg. length = {metrics['avg_length']:.3f}",
        transform=plt.gca().transAxes,
        verticalalignment="top",
        fontsize=10,
        bbox=dict(
            boxstyle="round",
            facecolor="white",
            alpha=0.88,
            edgecolor="gray",
        ),
    )

    plt.xlabel("Test sample index")
    plt.ylabel("ITE")
    plt.title(f"{method_name}: ITE interval vs true ITE, rep {iteration_id}")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.legend()
    plt.tight_layout()

    safe_name = (
        method_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("+", "plus")
    )

    path = os.path.join(
        output_dir,
        f"{safe_name}_ihdp_rep{iteration_id}.png",
    )

    plt.savefig(
        path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    return path, metrics


# =========================================================
# Cell 11: Evaluate one IHDP replication
# =========================================================

def evaluate_one_rep_config_with_tedvae(
    rep_id,
    cfg=CFG,
    device=DEVICE,
):
    print("\n" + "=" * 100)
    print(f"Running IHDP replication {rep_id}/100 | config_id={cfg['config_id']}")
    print("=" * 100)
    
    set_seed(GLOBAL_SEED + 1000 * cfg["config_id"] + rep_id)

    # -----------------------------------------------------
    # 11.1 Load IHDP data
    # -----------------------------------------------------
    data_train, data_test, contfeats, binfeats = IHDP(
        path="./IHDP_b/",
        reps=rep_id,
        cuda=False,
    )

    (z_train_raw, t_train_raw, y_train_raw), true_ite_train = data_train
    (z_test_raw, t_test_raw, y_test_raw), true_ite_test = data_test

    # Use only continuous features as proxy variables
    z_train_all = z_train_raw[:, contfeats].to(device).float()
    z_test = z_test_raw[:, contfeats].to(device).float()

    t_train_all = t_train_raw.to(device).float()
    y_train_all = y_train_raw.to(device).float()

    t_test = t_test_raw.to(device).float()
    y_test = y_test_raw.to(device).float()

    true_ite_train_np = to_numpy_1d(true_ite_train)
    true_ite_np = to_numpy_1d(true_ite_test)
    true_ate = float(np.mean(true_ite_np))

    # -----------------------------------------------------
    # 11.2 Standardize Z
    # -----------------------------------------------------
    if cfg["standardize_z"]:
        z_train_all, z_test, z_mean, z_std = standardize_train_test(
            z_train_all,
            z_test,
        )

    # -----------------------------------------------------
    # 11.3 Standardize Y only if cfg["standardize_y"] is True
    # -----------------------------------------------------
    if cfg["standardize_y"]:
        y_train_used, y_mean, y_std = standardize_y_train(y_train_all)
    else:
        y_train_used = y_train_all
        y_mean = torch.tensor(0.0, device=device)
        y_std = torch.tensor(1.0, device=device)

    # -----------------------------------------------------
    # 11.4 Build IntervalGP-VAE
    # -----------------------------------------------------
    use_aux = cfg["chosen_version"] == "u_aux"

    if cfg["chosen_version"] == "z_to_y":
        z_y_dim = z_train_all.shape[1]

    elif cfg["chosen_version"] == "split_z_to_t_and_y":
        z_y_dim = z_train_all.shape[1] // 2

    else:
        z_y_dim = None

    g_dl = torch.Generator()
    g_dl.manual_seed(GLOBAL_SEED + 5000 * cfg["config_id"] + rep_id)

    loader = DataLoader(
        TensorDataset(
            z_train_all,
            t_train_all,
            y_train_used,
        ),
        batch_size=cfg["batch_size"],
        shuffle=True,
        generator=g_dl,
        num_workers=0,
        persistent_workers=False,
    )

    model = CausalGPVAEwithNoise(
        input_dim=z_train_all.shape[1],
        hidden_dim=cfg["hidden_dim"],
        latent_dim=cfg["latent_dim"],
        head_hidden_dim=cfg["head_hidden_dim"],
        z_y_dim=z_y_dim,
        use_auxiliary_latents=use_aux,
        gp_lengthscale=cfg["gp_lengthscale"],
        gp_variance=cfg["gp_variance"],
        gp_noise=cfg["gp_noise"],
        kl_u_scale=cfg["kl_u_scale"],
        kl_eps_scale=cfg["kl_eps_scale"],
        kl_aux_scale=cfg["kl_aux_scale"],
        std_penalty_weight=cfg["std_penalty_weight"],
    ).to(device)

    start_time_intervalgpvae = time.time()

    model = train_intervalgpvae_config(
        model=model,
        loader=loader,
        cfg=cfg,
        device=device,
    )

    model.eval()

    # -----------------------------------------------------
    # 11.5 IntervalGP-VAE FullGP inference
    # Important:
    #   This seed matches grid-search config-specific inference.
    # -----------------------------------------------------
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(
            GLOBAL_SEED
            + 100000
            + 1000 * cfg["config_id"]
            + rep_id
        )

        intervalgp_results = ite_space_intervalgp_head(
            model=model,
            z_train=z_train_all,
            z_test=z_test,
            cfg=cfg,
        )

    intervalgp_ite_mean = intervalgp_results["ite_mean_test"]
    intervalgp_ite_lower = intervalgp_results["ite_lower_test"]
    intervalgp_ite_upper = intervalgp_results["ite_upper_test"]

    # If Y was standardized, transform ITE back to original Y scale.
    if cfg["standardize_y"]:
        intervalgp_ite_mean = intervalgp_ite_mean * y_std
        intervalgp_ite_lower = intervalgp_ite_lower * y_std
        intervalgp_ite_upper = intervalgp_ite_upper * y_std

    intervalgpvae_pehe = pehe_np(
        est_ite=intervalgp_ite_mean,
        true_ite=true_ite_np,
    )

    intervalgpvae_ate = float(
        np.mean(to_numpy_1d(intervalgp_ite_mean))
    )

    intervalgpvae_ate_error = abs(
        intervalgpvae_ate - true_ate
    )

    intervalgpvae_metrics = compute_interval_metrics(
        lower=intervalgp_ite_lower,
        upper=intervalgp_ite_upper,
        true_value=true_ite_np,
    )

    intervalgpvae_time = time.time() - start_time_intervalgpvae

    # -----------------------------------------------------
    # 11.6 TEDVAE baseline with conformal calibration
    # -----------------------------------------------------
    start_time_tedvae = time.time()

    z_train_cpu = z_train_all.detach().cpu().float()
    t_train_cpu = t_train_all.detach().cpu().float()
    y_train_cpu = y_train_all.detach().cpu().float()

    z_test_cpu = z_test.detach().cpu().float()

    fit_idx, cal_idx = split_train_calibration_indices(
        n=len(z_train_cpu),
        calibration_ratio=0.2,
        seed=123 + rep_id,
    )

    z_fit = z_train_cpu[fit_idx]
    t_fit = t_train_cpu[fit_idx]
    y_fit_raw = y_train_cpu[fit_idx]

    z_cal = z_train_cpu[cal_idx]

    true_ite_fit = true_ite_train_np[fit_idx]
    true_ite_cal = true_ite_train_np[cal_idx]

    ym = y_fit_raw.mean()
    ys = y_fit_raw.std()

    if ys.item() == 0:
        ys = torch.tensor(1.0)

    y_fit = (y_fit_raw - ym) / ys

    pyro.clear_param_store()

    # Fix TEDVAE seed for stable baseline comparison.
    set_seed(GLOBAL_SEED + 200000 + rep_id)

    tedvae = TEDVAE(
        feature_dim=z_fit.shape[1],
        continuous_dim=z_fit.shape[1],
        binary_dim=0,
        latent_dim=TEDVAE_ARGS["latent_dim"],
        latent_dim_t=TEDVAE_ARGS["latent_dim_t"],
        latent_dim_y=TEDVAE_ARGS["latent_dim_y"],
        hidden_dim=TEDVAE_ARGS["hidden_dim"],
        num_layers=TEDVAE_ARGS["num_layers"],
        num_samples=TEDVAE_ARGS["num_samples"],
    )

    tedvae.fit(
        z_fit,
        t_fit,
        y_fit,
        num_epochs=TEDVAE_ARGS["num_epochs"],
        batch_size=TEDVAE_ARGS["batch_size"],
        learning_rate=TEDVAE_ARGS["learning_rate"],
        learning_rate_decay=TEDVAE_ARGS["learning_rate_decay"],
        weight_decay=TEDVAE_ARGS["weight_decay"],
    )

    est_ite_fit = tedvae.ite(
        z_fit,
        ym,
        ys,
    )

    est_ite_cal = tedvae.ite(
        z_cal,
        ym,
        ys,
    )

    est_ite_test = tedvae.ite(
        z_test_cpu,
        ym,
        ys,
    )

    est_ite_fit_np = to_numpy_1d(est_ite_fit)
    est_ite_cal_np = to_numpy_1d(est_ite_cal)
    est_ite_test_np = to_numpy_1d(est_ite_test)

    q_hat_tedvae = conformal_abs_residual_quantile(
        y_true_cal=true_ite_cal,
        y_pred_cal=est_ite_cal_np,
        alpha=0.10,
    )

    tedvae_ite_lower, tedvae_ite_upper = make_conformal_interval(
        point_pred=est_ite_test_np,
        q_hat=q_hat_tedvae,
    )

    tedvae_metrics = compute_interval_metrics(
        lower=tedvae_ite_lower,
        upper=tedvae_ite_upper,
        true_value=true_ite_np,
    )

    tedvae_pehe_test = pehe_np(
        est_ite=est_ite_test_np,
        true_ite=true_ite_np,
    )

    tedvae_pehe_train = pehe_np(
        est_ite=est_ite_fit_np,
        true_ite=true_ite_fit,
    )

    tedvae_ate = float(
        np.mean(est_ite_test_np)
    )

    tedvae_ate_error = abs(
        tedvae_ate - true_ate
    )

    tedvae_time = time.time() - start_time_tedvae

    # -----------------------------------------------------
    # 11.7 Optional plots
    # -----------------------------------------------------
    if PLOT_EACH_REP:
        plot_ite_interval(
            true_ite=true_ite_np,
            ite_point=intervalgp_ite_mean,
            ite_lower=intervalgp_ite_lower,
            ite_upper=intervalgp_ite_upper,
            iteration_id=rep_id,
            method_name="IntervalGP-VAE FullGP Config",
            output_dir=OUTPUT_DIR,
        )

        plot_ite_interval(
            true_ite=true_ite_np,
            ite_point=est_ite_test_np,
            ite_lower=tedvae_ite_lower,
            ite_upper=tedvae_ite_upper,
            iteration_id=rep_id,
            method_name="TEDVAE Conformal",
            output_dir=OUTPUT_DIR,
        )

    # -----------------------------------------------------
    # 11.8 Save per-rep results
    # -----------------------------------------------------
    rep_df = pd.DataFrame({
        "rep_id": rep_id,
        "sample_index": np.arange(len(true_ite_np)),
        "true_ite": true_ite_np,

        "intervalgpvae_ite_mean": to_numpy_1d(intervalgp_ite_mean),
        "intervalgpvae_ite_lower": to_numpy_1d(intervalgp_ite_lower),
        "intervalgpvae_ite_upper": to_numpy_1d(intervalgp_ite_upper),

        "tedvae_ite_mean": est_ite_test_np,
        "tedvae_conformal_lower": tedvae_ite_lower,
        "tedvae_conformal_upper": tedvae_ite_upper,
    })

    rep_csv_path = os.path.join(
        OUTPUT_DIR,
        f"ihdp_rep_{rep_id}_config_results.csv",
    )

    rep_df.to_csv(
        rep_csv_path,
        index=False,
    )

    record = {
        "config_id": cfg["config_id"],
        "rep_id": rep_id,

        "intervalgpvae_pehe": intervalgpvae_pehe,
        "tedvae_pehe_test": tedvae_pehe_test,
        "tedvae_pehe_train": tedvae_pehe_train,

        "intervalgpvae_ate": intervalgpvae_ate,
        "tedvae_ate": tedvae_ate,
        "true_ate": true_ate,

        "intervalgpvae_ate_error": intervalgpvae_ate_error,
        "tedvae_ate_error": tedvae_ate_error,

        "intervalgpvae_coverage": intervalgpvae_metrics["coverage"],
        "intervalgpvae_interval_length": intervalgpvae_metrics["avg_length"],

        "tedvae_conformal_coverage": tedvae_metrics["coverage"],
        "tedvae_conformal_interval_length": tedvae_metrics["avg_length"],

        "intervalgpvae_time": intervalgpvae_time,
        "tedvae_time": tedvae_time,

        "rep_csv_path": rep_csv_path,
    }

    print("\n" + "-" * 80)
    print(f"IHDP replication {rep_id}: summary")
    print("-" * 80)
    print(f"IntervalGP-VAE config PEHE             : {intervalgpvae_pehe:.4f}")
    print(f"TEDVAE PEHE test                          : {tedvae_pehe_test:.4f}")
    print(f"TEDVAE PEHE train                         : {tedvae_pehe_train:.4f}")

    print(f"IntervalGP-VAE config ATE error        : {intervalgpvae_ate_error:.4f}")
    print(f"TEDVAE ATE error                          : {tedvae_ate_error:.4f}")

    print(f"IntervalGP-VAE config coverage         : {intervalgpvae_metrics['coverage']:.4f}")
    print(f"IntervalGP-VAE config interval length  : {intervalgpvae_metrics['avg_length']:.4f}")

    print(f"TEDVAE conformal coverage                 : {tedvae_metrics['coverage']:.4f}")
    print(f"TEDVAE conformal interval length          : {tedvae_metrics['avg_length']:.4f}")

    print(f"IntervalGP-VAE time                       : {intervalgpvae_time:.2f} s")
    print(f"TEDVAE time                               : {tedvae_time:.2f} s")
    print(f"Saved CSV                                 : {rep_csv_path}")
    print("-" * 80)

    return record


# =========================================================
# Cell 12: Run all replications
# =========================================================

all_records = []

for rep_id in SEARCH_REPS:
    record = evaluate_one_rep_config_with_tedvae(
        rep_id=rep_id,
        cfg=CFG,
        device=DEVICE,
    )

    all_records.append(record)

    # Save partial summary after each replication
    partial_df = pd.DataFrame(all_records)

    partial_summary_path = os.path.join(
        OUTPUT_DIR,
        "config_intervalgpvae_vs_tedvae_summary_partial.csv",
    )

    partial_df.to_csv(
        partial_summary_path,
        index=False,
    )

    print(f"Saved partial summary to: {partial_summary_path}")


# =========================================================
# Cell 13: Save final summary
# =========================================================

summary_df = pd.DataFrame(all_records)

summary_csv_path = os.path.join(
    OUTPUT_DIR,
    "config_intervalgpvae_vs_tedvae_summary_final.csv",
)

summary_df.to_csv(
    summary_csv_path,
    index=False,
)

print("\n" + "=" * 100)
print("Final average results for config")
print("=" * 100)

print(f"Avg IntervalGP-VAE config PEHE             : {summary_df['intervalgpvae_pehe'].mean():.4f}")
print(f"Avg TEDVAE PEHE test                          : {summary_df['tedvae_pehe_test'].mean():.4f}")
print(f"Avg TEDVAE PEHE train                         : {summary_df['tedvae_pehe_train'].mean():.4f}")

print(f"Avg IntervalGP-VAE config ATE error        : {summary_df['intervalgpvae_ate_error'].mean():.4f}")
print(f"Avg TEDVAE ATE error                          : {summary_df['tedvae_ate_error'].mean():.4f}")

print(f"Avg IntervalGP-VAE config coverage         : {summary_df['intervalgpvae_coverage'].mean():.4f}")
print(f"Avg IntervalGP-VAE config interval length  : {summary_df['intervalgpvae_interval_length'].mean():.4f}")

print(f"Avg TEDVAE conformal coverage                 : {summary_df['tedvae_conformal_coverage'].mean():.4f}")
print(f"Avg TEDVAE conformal interval length          : {summary_df['tedvae_conformal_interval_length'].mean():.4f}")

print(f"Avg IntervalGP-VAE time                       : {summary_df['intervalgpvae_time'].mean():.2f} s")
print(f"Avg TEDVAE time                               : {summary_df['tedvae_time'].mean():.2f} s")

print(f"Saved final summary CSV                       : {summary_csv_path}")
print("=" * 100)


# =========================================================
# Cell 14: Dashboard plots
# =========================================================

N = len(summary_df)

if N == 0:
    print("No results to plot.")

else:
    x = np.arange(1, N + 1)

    # -----------------------------------------------------
    # Plot 1: PEHE
    # -----------------------------------------------------
    plt.figure(figsize=(12, 5))

    plt.plot(
        x,
        summary_df["intervalgpvae_pehe"],
        marker="o",
        linestyle="-",
        label=f"IntervalGP-VAE config, mean={summary_df['intervalgpvae_pehe'].mean():.4f}",
    )

    plt.plot(
        x,
        summary_df["tedvae_pehe_test"],
        marker="s",
        linestyle="-",
        label=f"TEDVAE, mean={summary_df['tedvae_pehe_test'].mean():.4f}",
    )

    plt.xlabel("IHDP replication")
    plt.ylabel("PEHE")
    plt.title("IHDP PEHE: IntervalGP-VAE config vs TEDVAE")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()

    pehe_plot_path = os.path.join(
        OUTPUT_DIR,
        "config_ihdp_pehe_comparison.png",
    )

    plt.savefig(
        pehe_plot_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    # -----------------------------------------------------
    # Plot 2: ATE error
    # -----------------------------------------------------
    plt.figure(figsize=(12, 5))

    plt.plot(
        x,
        summary_df["intervalgpvae_ate_error"],
        marker="o",
        linestyle="-",
        label=f"IntervalGP-VAE config, mean={summary_df['intervalgpvae_ate_error'].mean():.4f}",
    )

    plt.plot(
        x,
        summary_df["tedvae_ate_error"],
        marker="s",
        linestyle="-",
        label=f"TEDVAE, mean={summary_df['tedvae_ate_error'].mean():.4f}",
    )

    plt.xlabel("IHDP replication")
    plt.ylabel("ATE error")
    plt.title("IHDP ATE Error: IntervalGP-VAE config vs TEDVAE")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()

    ate_plot_path = os.path.join(
        OUTPUT_DIR,
        "config_ihdp_ate_error_comparison.png",
    )

    plt.savefig(
        ate_plot_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    # -----------------------------------------------------
    # Plot 3: Coverage
    # -----------------------------------------------------
    plt.figure(figsize=(12, 5))

    plt.plot(
        x,
        summary_df["intervalgpvae_coverage"],
        marker="o",
        linestyle="-",
        label=f"IntervalGP-VAE config, mean={summary_df['intervalgpvae_coverage'].mean():.4f}",
    )

    plt.plot(
        x,
        summary_df["tedvae_conformal_coverage"],
        marker="s",
        linestyle="-",
        label=f"TEDVAE conformal, mean={summary_df['tedvae_conformal_coverage'].mean():.4f}",
    )

    plt.axhline(
        0.90,
        linestyle="--",
        alpha=0.7,
        label="Nominal 90%",
    )

    plt.xlabel("IHDP replication")
    plt.ylabel("Coverage")
    plt.title("IHDP ITE Interval Coverage")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()

    coverage_plot_path = os.path.join(
        OUTPUT_DIR,
        "config_ihdp_coverage_comparison.png",
    )

    plt.savefig(
        coverage_plot_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    # -----------------------------------------------------
    # Plot 4: Interval length
    # -----------------------------------------------------
    plt.figure(figsize=(12, 5))

    plt.plot(
        x,
        summary_df["intervalgpvae_interval_length"],
        marker="o",
        linestyle="-",
        label=f"IntervalGP-VAE config, mean={summary_df['intervalgpvae_interval_length'].mean():.4f}",
    )

    plt.plot(
        x,
        summary_df["tedvae_conformal_interval_length"],
        marker="s",
        linestyle="-",
        label=f"TEDVAE conformal, mean={summary_df['tedvae_conformal_interval_length'].mean():.4f}",
    )

    plt.xlabel("IHDP replication")
    plt.ylabel("Average interval length")
    plt.title("IHDP ITE Interval Length")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()

    length_plot_path = os.path.join(
        OUTPUT_DIR,
        "config_ihdp_interval_length_comparison.png",
    )

    plt.savefig(
        length_plot_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    print("\nSaved plots:")
    print(pehe_plot_path)
    print(ate_plot_path)
    print(coverage_plot_path)
    print(length_plot_path)

# end

# plot

In [ ]:
# =========================================================
# Cell 14: Dashboard plots
# =========================================================

N = len(summary_df)

if N == 0:
    print("No results to plot.")
else:
    x = np.arange(1, N + 1)
    
    # 获取均值用于绘制参考线和标签
    mean_gpvae_pehe = summary_df["intervalgpvae_pehe"].mean()
    mean_tedvae_pehe = summary_df["tedvae_pehe_test"].mean()
    
    mean_gpvae_ate = summary_df["intervalgpvae_ate_error"].mean()
    mean_tedvae_ate = summary_df["tedvae_ate_error"].mean()
    
    mean_gpvae_cvg = summary_df["intervalgpvae_coverage"].mean()
    mean_tedvae_cvg = summary_df["tedvae_conformal_coverage"].mean()
    
    mean_gpvae_len = summary_df["intervalgpvae_interval_length"].mean()
    mean_tedvae_len = summary_df["tedvae_conformal_interval_length"].mean()

    COLOR_GPVAE = "tab:orange"
    COLOR_TEDVAE = "tab:blue"

    bar_width = 0.35
    indices = np.arange(len(x))

    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    
    # -----------------------------------------------------
    # Plot 1 (Top-Left): PEHE 
    # -----------------------------------------------------
    ax1 = axes[0, 0]
    ax1.plot(x, summary_df["intervalgpvae_pehe"], marker="o", markersize=5, color=COLOR_GPVAE, 
             linestyle="-", linewidth=2.0, alpha=0.85, label="IntervalGP-VAE")
    ax1.plot(x, summary_df["tedvae_pehe_test"], marker="s", markersize=5, color=COLOR_TEDVAE, 
             linestyle="-", linewidth=2.0, alpha=0.85, label="Conformalized TEDVAE")
    
    ax1.axhline(mean_gpvae_pehe, color=COLOR_GPVAE, linestyle="--", linewidth=2.5,
                label=f"IntervalGP-VAE Mean: {mean_gpvae_pehe:.4f}")
    ax1.axhline(mean_tedvae_pehe, color=COLOR_TEDVAE, linestyle="--", linewidth=2.5,
                label=f"Conformalized TEDVAE Mean: {mean_tedvae_pehe:.4f}")

    ax1.set_xlabel("IHDP replication", fontsize=11)
    ax1.set_ylabel("PEHE", fontsize=12, fontweight='bold')
    ax1.legend(loc="upper right", framealpha=0.9, fontsize=9)
    ax1.grid(True, linestyle="--", alpha=0.5)

    # -----------------------------------------------------
    # Plot 2 (Top-Right): ATE Error
    # -----------------------------------------------------
    ax2 = axes[0, 1]
    ax2.plot(x, summary_df["intervalgpvae_ate_error"], marker="o", markersize=5, color=COLOR_GPVAE, 
             linestyle="-", linewidth=2.0, alpha=0.85, label="IntervalGP-VAE")
    ax2.plot(x, summary_df["tedvae_ate_error"], marker="s", markersize=5, color=COLOR_TEDVAE, 
             linestyle="-", linewidth=2.0, alpha=0.85, label="Conformalized TEDVAE")
    
    ax2.axhline(mean_gpvae_ate, color=COLOR_GPVAE, linestyle="--", linewidth=2.5,
                label=f"IntervalGP-VAE Mean: {mean_gpvae_ate:.4f}")
    ax2.axhline(mean_tedvae_ate, color=COLOR_TEDVAE, linestyle="--", linewidth=2.5,
                label=f"Conformalized TEDVAE Mean: {mean_tedvae_ate:.4f}")

    ax2.set_xlabel("IHDP replication", fontsize=11)
    ax2.set_ylabel("ATE Error", fontsize=12, fontweight='bold')
    ax2.legend(loc="upper right", framealpha=0.9, fontsize=9)
    ax2.grid(True, linestyle="--", alpha=0.5)

    # -----------------------------------------------------
    # Plot 3 (Bottom-Left): Coverage 
    # -----------------------------------------------------
    ax3 = axes[1, 0]
    ax3.bar(indices - bar_width/2, summary_df["intervalgpvae_coverage"], 
            width=bar_width, color=COLOR_GPVAE, alpha=0.9, label=f"IntervalGP-VAE (Avg: {mean_gpvae_cvg:.3f})")
    ax3.bar(indices + bar_width/2, summary_df["tedvae_conformal_coverage"], 
            width=bar_width, color=COLOR_TEDVAE, alpha=0.9, label=f"Conformalized TEDVAE (Avg: {mean_tedvae_cvg:.3f})")
    
    ax3.axhline(0.90, color="red", linestyle="-.", alpha=0.8, linewidth=2.0, label="Nominal 90%")

    ax3.set_xlabel("IHDP replication index", fontsize=11)
    ax3.set_ylabel("ITE Interval Coverage Rate", fontsize=12, fontweight='bold')
    ax3.set_xticks(indices[::5])
    ax3.set_xticklabels(x[::5])  
    ax3.set_ylim(0, 1.05)
    ax3.legend(loc="lower right", framealpha=0.9, fontsize=9)
    ax3.grid(True, linestyle="--", alpha=0.4, axis="y")

    # -----------------------------------------------------
    # Plot 4 (Bottom-Right): Interval Length
    # -----------------------------------------------------
    ax4 = axes[1, 1]
    ax4.bar(indices - bar_width/2, summary_df["intervalgpvae_interval_length"], 
            width=bar_width, color=COLOR_GPVAE, alpha=0.9, label=f"IntervalGP-VAE (Avg: {mean_gpvae_len:.3f})")
    ax4.bar(indices + bar_width/2, summary_df["tedvae_conformal_interval_length"], 
            width=bar_width, color=COLOR_TEDVAE, alpha=0.9, label=f"Conformalized TEDVAE (Avg: {mean_tedvae_len:.3f})")

    ax4.set_xlabel("IHDP replication index", fontsize=11)
    ax4.set_ylabel("ITE Interval Average Length", fontsize=12, fontweight='bold')
    ax4.set_xticks(indices[::5])
    ax4.set_xticklabels(x[::5])  
    ax4.legend(loc="upper right", framealpha=0.9, fontsize=9)
    ax4.grid(True, linestyle="--", alpha=0.4, axis="y")
    plt.tight_layout()
    
    dashboard_plot_path = os.path.join(OUTPUT_DIR, "ihdp_results_dashboard_2x2_conformalized.png")
    plt.savefig(dashboard_plot_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"\nSaved updated 2x2 dashboard plot to:\n{dashboard_plot_path}")

# end